In [1]:
from pydub import AudioSegment
import os
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pyloudnorm as pyln


In [2]:
from pathlib import Path

# --- Paths ---
INPUT_DIR  = Path(r"C:\Users\Dhanuja\Downloads\dataset\instrumentals")     
OUTPUT_DIR = Path(r"C:\Users\Dhanuja\Downloads\dataset\chunked_dataset")

# --- Audio ---
TARGET_SR      = 44100
CHANNELS       = 1        # DAC expects mono

# --- Chunking ---
CHUNK_MS       = 4000
OVERLAP_MS     = 2000
STEP_MS        = CHUNK_MS - OVERLAP_MS
MIN_TRACK_MS   = 10_000   # skip tracks shorter than 10s

# --- Loudness ---
TARGET_LUFS    = -14.0    # standard for music, safe for DAC input
LUFS_CEILING   = -1.0     # true peak ceiling to avoid clipping post-norm

In [3]:
def load_and_validate(filepath: Path) -> AudioSegment | None:
    try:
        audio = AudioSegment.from_file(filepath)
    except Exception as e:
        print(f"[SKIP] Could not load {filepath.name}: {e}")
        return None

    if len(audio) < MIN_TRACK_MS:
        print(f"[SKIP] Too short ({len(audio)/1000:.1f}s): {filepath.name}")
        return None

    return audio  # already mono @ 44100, trust the source

In [4]:


def normalize_lufs(audio: AudioSegment, target_lufs: float, ceiling_db: float) -> AudioSegment:
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / 32768.0
    
    meter = pyln.Meter(TARGET_SR)
    
    # Measure loudness — pyloudnorm needs (samples, channels)
    loudness = meter.integrated_loudness(samples)  # 1D, no reshape needed
    
    # If loudness is -inf (silence/very quiet), skip normalization
    if not np.isfinite(loudness):
        print("[WARN] Could not measure loudness, skipping normalization")
        return audio
    
    # Normalize to target LUFS
    normalized = pyln.normalize.loudness(samples, loudness, target_lufs)
    
    # True peak ceiling — prevent clipping
    peak = np.max(np.abs(normalized))
    ceiling_linear = 10 ** (ceiling_db / 20)
    if peak > ceiling_linear:
        normalized = normalized * (ceiling_linear / peak)
    
    # Convert back to pydub AudioSegment
    pcm = (normalized * 32768.0).astype(np.int16)
    return AudioSegment(
        pcm.tobytes(),
        frame_rate=TARGET_SR,
        sample_width=2,
        channels=CHANNELS
    )

In [5]:
def chunk_audio(audio: AudioSegment) -> list[tuple[int, AudioSegment]]:
    chunks = []
    start_ms = 0

    while start_ms + CHUNK_MS <= len(audio):
        chunk = audio[start_ms : start_ms + CHUNK_MS]
        chunks.append((start_ms, chunk))
        start_ms += STEP_MS

    return chunks

In [6]:


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUPPORTED = {".mp3", ".wav", ".flac", ".ogg"}

files = [f for f in INPUT_DIR.rglob("*") if f.suffix.lower() in SUPPORTED]
print(f"Found {len(files)} tracks")

total_chunks = 0
skipped = 0

for filepath in tqdm(files, desc="Processing"):
    audio = load_and_validate(filepath)
    if audio is None:
        skipped += 1
        continue

    audio = normalize_lufs(audio, TARGET_LUFS, LUFS_CEILING)
    chunks = chunk_audio(audio)

    # Mirror folder structure
    out_folder = OUTPUT_DIR / filepath.stem
    out_folder.mkdir(parents=True, exist_ok=True)

    for i, (start_ms, chunk) in enumerate(chunks):
        out_name = f"{filepath.stem}_chunk{i:03d}_{start_ms}ms.mp3"
        chunk.export(
            out_folder / out_name,
            format="mp3",
            bitrate="320k"       
        )
        total_chunks += 1

print(f"\nDone. {total_chunks} chunks saved, {skipped} tracks skipped.")

Found 2036 tracks


Processing:   0%|          | 1/2036 [00:08<4:37:53,  8.19s/it]c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\.venv\lib\site-packages\pyloudnorm\normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Processing:   0%|          | 2/2036 [00:11<3:06:02,  5.49s/it]c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\.venv\lib\site-packages\pyloudnorm\normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Processing:   0%|          | 3/2036 [00:15<2:36:14,  4.61s/it]c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\.venv\lib\site-packages\pyloudnorm\normalize.py:62: UserWarning: Possible clipped samples in output.
  warnings.warn("Possible clipped samples in output.")
Processing:   0%|          | 5/2036 [00:22<2:15:59,  4.02s/it]c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\.venv\lib\site-packages\pyloudnorm\normalize.py:62: UserWarning: Possible clipped samples in output.
  w

KeyboardInterrupt: 

In [ ]:
all_chunks = list(OUTPUT_DIR.rglob("*.mp3"))
print(f"Total chunks on disk: {len(all_chunks)}")
print(f"Expected ~{2036 * 14} from full 30s tracks")

# Spot check one chunk
test = AudioSegment.from_file(all_chunks[0])
print(f"Sample chunk — duration: {len(test)/1000}s | sr: {test.frame_rate} | channels: {test.channels}")


In [7]:
import pyloudnorm as pyln
import numpy as np
from pydub import AudioSegment
from pathlib import Path

meter = pyln.Meter(44100)
loudness_values = []

for f in list(INPUT_DIR.rglob("*.mp3"))[:100]:  # sample 100 tracks
    audio = AudioSegment.from_file(f)
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / 32768.0
    l = meter.integrated_loudness(samples)
    if np.isfinite(l):
        loudness_values.append(l)

print(f"Mean: {np.mean(loudness_values):.1f} LUFS")
print(f"Std:  {np.std(loudness_values):.1f} LU")
print(f"Min:  {np.min(loudness_values):.1f} LUFS")
print(f"Max:  {np.max(loudness_values):.1f} LUFS")

Mean: -19.8 LUFS
Std:  4.4 LU
Min:  -32.5 LUFS
Max:  -10.7 LUFS


In [8]:
# Check the 5 quietest tracks specifically
import pyloudnorm as pyln
import numpy as np
from pydub import AudioSegment
from pathlib import Path

meter = pyln.Meter(44100)
results = []

for f in INPUT_DIR.rglob("*.mp3"):
    audio = AudioSegment.from_file(f)
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / 32768.0
    l = meter.integrated_loudness(samples)
    if np.isfinite(l):
        results.append((l, f.name, len(audio)/1000))

results.sort()
for loudness, name, duration in results[:5]:
    print(f"{loudness:.1f} LUFS | {duration:.1f}s | {name}")

KeyboardInterrupt: 